# **Extracting Information from Legal Documents Using RAG**

## **Objective**

The main objective of this assignment is to process and analyse a collection text files containing legal agreements (e.g., NDAs) to prepare them for implementing a **Retrieval-Augmented Generation (RAG)** system. This involves:

* Understand the Cleaned Data : Gain a comprehensive understanding of the structure, content, and context of the cleaned dataset.
* Perform Exploratory Analysis : Conduct bivariate and multivariate analyses to uncover relationships and trends within the cleaned data.
* Create Visualisations : Develop meaningful visualisations to support the analysis and make findings interpretable.
* Derive Insights and Conclusions : Extract valuable insights from the cleaned data and provide clear, actionable conclusions.
* Document the Process : Provide a detailed description of the data, its attributes, and the steps taken during the analysis for reproducibility and clarity.

The ultimate goal is to transform the raw text data into a clean, structured, and analysable format that can be effectively used to build and train a RAG system for tasks like information retrieval, question-answering, and knowledge extraction related to legal agreements.

### **Business Value**  


The project aims to leverage RAG to enhance legal document processing for businesses, law firms, and regulatory bodies. The key business objectives include:

* Faster Legal Research: <br> Reduce the time lawyers and compliance officers spend searching for relevant case laws, precedents, statutes, or contract clauses.
* Improved Contract Analysis: <br> Automatically extract key terms, obligations, and risks from lengthy contracts.
* Regulatory Compliance Monitoring: <br> Help businesses stay updated with legal and regulatory changes by retrieving relevant legal updates.
* Enhanced Decision-Making: <br> Provide accurate and context-aware legal insights to assist in risk assessment and legal strategy.


**Use Cases**
* Legal Chatbots
* Contract Review Automation
* Tracking Regulatory Changes and Compliance Monitoring
* Case Law Analysis of past judgments
* Due Diligence & Risk Assessment

## **1. Data Loading, Preparation and Analysis** <font color=red> [20 marks] </font><br>

### **1.1 Data Understanding**

The dataset contains legal documents and contracts collected from various sources. The documents are present as text files (`.txt`) in the *corpus* folder.

There are four types of documents in the *courpus* folder, divided into four subfolders.
- `contractnli`: contains various non-disclosure and confidentiality agreements
- `cuad`: contains contracts with annotated legal clauses
- `maud`: contains various merger/acquisition contracts and agreements
- `privacy_qa`: a question-answering dataset containing privacy policies

The dataset also contains evaluation files in JSON format in the *benchmark* folder. The files contain the questions and their answers, along with sources. For the above folders, there is a `json` file: `contractnli.json`, `cuad.json`, `maud.json`. The file structure is as follows:

```
{
    "tests": [
        {
            "query": <question1>,
            "snippets": [{
                    "file_path": <source_file1>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 1>
                },
                {
                    "file_path": <source_file2>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 2>
                }, ....
            ]
        },
        {
            "query": <question2>,
            "snippets": [{<answer context for que 2>}]
        },
        ... <more queries>
    ]
}
```

### **1.2 Load and Preprocess the data** <font color=red> [5 marks] </font><br>

#### Loading libraries

In [ ]:
## The following libraries might be useful
!pip install -q langchain-openai
!pip install -U -q langchain-community
!pip install -U -q langchain-chroma
!pip install -U -q datasets
!pip install -U -q ragas
!pip install -U -q rouge_score

In [ ]:
# Import essential libraries
import os
from collections import Counter

from langchain_community.document_loaders import DirectoryLoader, TextLoader


#### **1.2.1** <font color=red> [3 marks] </font>
Load all `.txt` files from the folders.

You can utilise document loaders from the options provided by the LangChain community.

Optionally, you can also read the files manually, while ensuring proper handling of encoding issues (e.g., utf-8, latin1). In such case, also store the file content along with metadata (e.g., file name, directory path) for traceability.

In [ ]:
# Get the filepath from environment variable or set it directly in the code. The corpus directory contains the text files to be loaded as documents.
# CORPUS_DIR = r"C:\Ritu Data\Upgrad\GenAI\RAG_Assignment\rag_legal\corpus"

# load environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

CORPUS_DIR = os.environ.get("CORPUS_DIR")
PERSIST_DIR = os.environ.get("PERSIST_DIR")
BENCHMARK_DIR = os.environ.get("BENCHMARK_DIR")
print(f"CORPUS_DIR: {CORPUS_DIR}")
print(f"PERSIST_DIR: {PERSIST_DIR}")
print(f"BENCHMARK_DIR: {BENCHMARK_DIR}")

In [ ]:
# Load the files as documents
# Use document loaders to load the text files as documents. You can use the TextLoader from langchain.document_loaders to load the text files.


def load_files_as_documents(corpus_dir):
    """Recursively load all .txt files with LangChain's DirectoryLoader + TextLoader.
    autodetect_encoding=True tries to detect the encoding (utf-8, latin-1, ...) when a file
    can't be read as utf-8; silent_errors=True skips unreadable files instead of stopping."""
    loader = DirectoryLoader(
        corpus_dir,
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"autodetect_encoding": True},
        recursive=True,
        silent_errors=True,
        show_progress=True,
    )
    documents = loader.load()

    # Add metadata for traceability
    for document in documents:
        file_path = document.metadata["source"]
        document.metadata["file_name"] = os.path.basename(file_path)
        document.metadata["folder"] = os.path.basename(os.path.dirname(file_path))   # contractnli / cuad / maud / privacy_qa
        document.metadata["rel_path"] = os.path.relpath(file_path, corpus_dir)
    return documents

docs = load_files_as_documents(CORPUS_DIR)

# Sanity checks
print(f"Loaded {len(docs)} documents")
print(Counter(document.metadata["folder"] for document in docs))
print(f"Source: {docs[0].metadata['file_name']}, Path: {docs[0].metadata['rel_path']}")
print(f"Page Content: {docs[0].page_content[:300]}")


#### **1.2.2** <font color=red> [2 marks] </font>
Preprocess the text data to remove noise and prepare it for analysis.

Remove special characters, extra whitespace, and irrelevant content such as email and telephone contact info.
Normalise text (e.g., convert to lowercase, remove stop words).
Handle missing or corrupted data by logging errors and skipping problematic files.

In [ ]:
 !pip install -q nltk

In [ ]:
# Clean and preprocess the data
import re
import logging
import nltk
from nltk.corpus import stopwords
from langchain_core.documents import Document

nltk.download("stopwords", quiet=True)
STOP_WORDS = set(stopwords.words("english"))

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("preprocess")

EMAIL_RE = re.compile(r"[\w\.\-+]+@[\w\-]+(?:\.[\w\-]+)+")
URL_RE = re.compile(r"(?:https?://|www\.)\S+")
PHONE_RE = re.compile(r"(?:\+?\d{1,3}[\s.\-]?)?(?:\(\d{2,4}\)[\s.\-]?)?\d{3,4}[\s.\-]\d{3,4}(?:[\s.\-]\d{2,4})?")
SPECIAL_RE = re.compile(r"[^a-z0-9\s]")
SPACE_RE = re.compile(r"\s+")

def clean_text(text, remove_stopwords=True):
    """Remove emails/URLs/phone numbers and special characters, normalise to lowercase,
    collapse whitespace and (optionally) drop stop words."""
    text = EMAIL_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = PHONE_RE.sub(" ", text)
    text = text.lower()
    text = SPECIAL_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    if remove_stopwords:
        text = " ".join(word for word in text.split() if word not in STOP_WORDS)
    return text

def preprocess_documents(documents):
    """Return cleaned copies of the documents. Empty/corrupted ones are logged and skipped.
    The original text is kept in metadata['original_text'] (useful for chunking/benchmark spans)."""
    cleaned_docs, skipped_files = [], []
    for document in documents:
        source_path = document.metadata.get("source", "unknown")
        try:
            if document.page_content is None or not document.page_content.strip():
                raise ValueError("empty document")
            cleaned_text = clean_text(document.page_content)
            if not cleaned_text:
                raise ValueError("no content left after cleaning")
            metadata = dict(document.metadata)
            metadata["original_text"] = document.page_content
            cleaned_docs.append(Document(page_content=cleaned_text, metadata=metadata))
        except Exception as error:
            logger.error("Skipping %s: %s", source_path, error)
            skipped_files.append(source_path)
    print(f"Cleaned {len(cleaned_docs)} documents; skipped {len(skipped_files)}")
    return cleaned_docs

cleaned_docs = preprocess_documents(docs)

# Before/after check
print("BEFORE:", docs[0].page_content[:300])
print("AFTER :", cleaned_docs[0].page_content[:300])


### **1.3 Exploratory Data Analysis** <font color=red> [10 marks] </font><br>

#### **1.3.1** <font color=red> [2 marks] </font>
Calculate the average, maximum and minimum document length.

In [ ]:
# Calculate the average, maximum and minimum document length.
import pandas as pd

# Length in words (cleaned text) and in characters (original text)
lengths = pd.DataFrame({
    "file_name": [document.metadata["file_name"] for document in cleaned_docs],
    "folder": [document.metadata["folder"] for document in cleaned_docs],
    "word_count": [len(document.page_content.split()) for document in cleaned_docs],
    "char_count": [len(document.metadata["original_text"]) for document in cleaned_docs],
})

# Overall statistics
print("Document length (words, after cleaning):")
print(f"  Average: {lengths['word_count'].mean():.1f}")
print(f"  Maximum: {lengths['word_count'].max()}  ({lengths.loc[lengths['word_count'].idxmax(), 'file_name']})")
print(f"  Minimum: {lengths['word_count'].min()}  ({lengths.loc[lengths['word_count'].idxmin(), 'file_name']})")

print("\nDocument length (characters, original text):")
print(f"  Average: {lengths['char_count'].mean():.1f}")
print(f"  Maximum: {lengths['char_count'].max()}")
print(f"  Minimum: {lengths['char_count'].min()}")

# Breakdown by document type (folder)
print("\nWord count by folder:")
print(lengths.groupby("folder")["word_count"].agg(["count", "mean", "max", "min"]).round(1))


#### **1.3.2** <font color=red> [4 marks] </font>
Analyse the frequency of occurence of words and find the most and least occuring words.

Find the 20 most common and least common words in the text. Ignore stop words such as articles and prepositions.

In [ ]:
# Find frequency of occurence of words
from collections import Counter
import matplotlib.pyplot as plt

# Count words across all cleaned documents (stop words are already removed).
# Keep only alphabetic tokens of 3+ letters to avoid numbers and leftover fragments.
word_counts = Counter()
for document in cleaned_docs:
    word_counts.update(word for word in document.page_content.split() if word.isalpha() and len(word) >= 3)

print(f"Total words: {sum(word_counts.values())}, unique words: {len(word_counts)}")

# 20 most common words
most_common = word_counts.most_common(20)
print("\n20 most common words:")
for word, count in most_common:
    print(f"  {word:<20}{count}")

# 20 least common words (many words appear exactly once, so ties are broken alphabetically)
least_common = sorted(word_counts.items(), key=lambda word_and_count: (word_and_count[1], word_and_count[0]))[:20]
print("\n20 least common words:")
for word, count in least_common:
    print(f"  {word:<20}{count}")

hapax = sum(1 for count in word_counts.values() if count == 1)
print(f"\nWords that occur only once: {hapax} ({hapax / len(word_counts):.1%} of vocabulary)")

# Bar chart of the most common words
top_words, top_counts = zip(*most_common)
plt.figure(figsize=(10, 5))
plt.bar(top_words, top_counts, color="steelblue")
plt.xticks(rotation=60, ha="right")
plt.title("20 Most Common Words in the Corpus")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


#### **1.3.3** <font color=red> [4 marks] </font>
Analyse the similarity of different documents to each other based on TF-IDF vectors.

Transform some documents to TF-IDF vectors and calculate their similarity matrix using a suitable distance function. If contracts contain duplicate or highly similar clauses, similarity calculation can help detect them.

Identify for the first 10 documents and then for 10 random documents. What do you observe?

In [ ]:
# Transform the page contents of documents
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Fit TF-IDF on the whole cleaned corpus so IDF weights reflect all documents
texts = [document.page_content for document in cleaned_docs]
names = [document.metadata["file_name"][:25] for document in cleaned_docs]

tfidf = TfidfVectorizer(max_features=20000)
tfidf_matrix = tfidf.fit_transform(texts)
print("TF-IDF matrix shape:", tfidf_matrix.shape)

def similarity_report(indices, title):
    """Cosine similarity matrix for the given documents, shown as a heatmap."""
    similarity_matrix = cosine_similarity(tfidf_matrix[indices])
    labels = [f"{doc_idx}: {names[doc_idx]}" for doc_idx in indices]
    similarity_df = pd.DataFrame(similarity_matrix, index=labels, columns=indices)
    plt.figure(figsize=(9, 7))
    sns.heatmap(similarity_df, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
    plt.title(title)
    plt.tight_layout()
    plt.show()

    # Most similar pair (excluding the diagonal)
    off_diagonal = similarity_matrix.copy()
    np.fill_diagonal(off_diagonal, -1)
    first_idx, second_idx = np.unravel_index(off_diagonal.argmax(), off_diagonal.shape)
    print(f"Most similar pair: {labels[first_idx]}  <->  {labels[second_idx]}  (score {off_diagonal[first_idx, second_idx]:.2f})")
    print(f"Average pairwise similarity: {off_diagonal[off_diagonal >= 0].mean():.2f}")
    return similarity_matrix

# Compute similarity scores
# Compute similarity scores for the first 10 documents
# first_10 = list(range(10))
# sim_first = similarity_report(first_10, "Cosine similarity - first 10 documents")


In [ ]:
# Compute similarity scores
# Compute similarity scores for the first 10 documents
first_10 = list(range(10))
sim_first = similarity_report(first_10, "Cosine similarity - first 10 documents")

In [ ]:
# create a list of 10 random integers
import random
random.seed(42)
# random_10 = random.sample(range(len(cleaned_docs)), 10)
# print(random_10)


In [ ]:
# Compute similarity scores for 10 random documents
sim_random = similarity_report(random_10, "Cosine similarity - 10 random documents")


#### **Observations**

**First 10 documents (all `contractnli` NDAs)**
- Similarity is high and consistent: the average pairwise cosine score is about 0.45, and most pairs score between 0.4 and 0.75.
- The most similar pair is `5-NSK-Confidentiality-Agreement-for-Suppliers` and `Aspiegel_NDA_template` (about 0.76). NDAs share the same standard clauses, such as definition of confidential information, obligations, term, return of materials and governing law. They are written from near-identical templates, so they use much of the same vocabulary.
- The exceptions are the last document (about 0.13 to 0.21 against every other) and, to a lesser degree, the eighth (about 0.22 to 0.41). These documents are probably structured differently, for example as a data-use or policy-style agreement, or they have unusual wording.

**10 random documents (mixed `cuad`, `maud` and `contractnli`)**
- Similarity is very low: the average pairwise score is about 0.07, and almost every pair is under 0.10. The documents come from different types of agreements (distribution, endorsement, joint venture, merger, NDA), so their vocabulary has little in common.
- The one exception is a pair of `maud` merger agreements, `Eidos Therapeutics / BridgeBio Pharma` and `People's United Financial / M&T Bank`, at about 0.87. They are the same kind of document, and merger agreements reuse long boilerplate sections.

**Takeaways**
- TF-IDF similarity picks out documents of the same type. NDAs are similar to each other, and merger agreements are similar to each other, while mixed contract types are not.
- Highly similar documents share duplicated or near-duplicated clauses. For RAG, this means retrieval may return near-identical chunks from different contracts. That argues for metadata filtering by document or source, deduplication, or re-ranking.
- The contrast between the two samples shows that the corpus is clustered by document type (`contractnli`, `cuad`, `maud`, `privacy_qa`). Retrieval quality may therefore depend on which kind of question is asked.


### **1.4 Document Creation and Chunking** <font color=red> [5 marks] </font><br>

#### **1.4.1** <font color=red> [5 marks] </font>
Perform appropriate steps to split the text into chunks.

In [ ]:
# SUPERSEDED by sections 2.3-2.5 (old fixed-size chunking (2000/500)); kept for reference
# # Process files and generate chunks
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# # Split the original (uncleaned) documents: embeddings and answer generation work better on
# # natural text, and the benchmark answer spans refer to the original text.
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,
#     chunk_overlap=200,
#     separators=["\n\n", "\n", ". ", " ", ""],
#     add_start_index=True,      # stores the character offset of each chunk in metadata["start_index"]
# )

# chunks = text_splitter.split_documents(docs)

# # Sanity checks
# chunk_lengths = [len(chunk.page_content) for chunk in chunks]
# print(f"{len(docs)} documents -> {len(chunks)} chunks")
# print(f"Chunk length (chars): avg {sum(chunk_lengths)/len(chunk_lengths):.0f}, max {max(chunk_lengths)}, min {min(chunk_lengths)}")
# print(Counter(chunk.metadata["folder"] for chunk in chunks))
# print(chunks[0].metadata)
# print(chunks[0].page_content[:300])


## **2. Vector Database and RAG Chain Creation** <font color=red> [15 marks] </font><br>

### **2.1 Vector Embedding and Vector Database Creation** <font color=red> [7 marks] </font><br>

#### **2.1.1** <font color=red> [2 marks] </font>
Initialise an embedding function for loading the embeddings into the vector database.

Initialise a function to transform the text to vectors using an embedding model. You can also use this function to transform during vector DB creation itself.

In [ ]:
# Fetching API Key as an environment variable (or load it directly if variable naming is conventional)
import os
from getpass import getpass

# Set OPENAI_API_KEY in environment beforehand; if it is missing you are asked for it here.
# Never paste the key into the notebook itself.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

In [ ]:
# Initialise an embedding function
from langchain_openai import OpenAIEmbeddings

embedding_function = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 1536 dimensions for text-embedding-3-small, 3072 for text-embedding-3-large
    openai_api_key=OPENAI_API_KEY,
)

test_vector = embedding_function.embed_query("confidentiality obligations of the receiving party")
print(len(test_vector))   # 1536 dimensions for text-embedding-3-small


#### **2.1.2** <font color=red> [5 marks] </font>
Load the embeddings to a vector database.

Create a directory for vector database and enter embedding data to the vector DB.

In [ ]:
# SUPERSEDED by sections 2.3-2.5 (old Chroma DB creation and chunk ids); kept for reference
# # Add Chunks to vector DB
# from langchain_chroma import Chroma
# from tqdm.auto import tqdm

# # Create a directory for the vector database
# # Take PERSIST_DIR as an environment variable or set it directly in the code. The vector database will be stored in this directory.
# # PERSIST_DIR = os.environ.get("PERSIST_DIR", r"C:\Ritu Data\Upgrad\GenAI\RAG_Assignment\rag_legal\chroma_db_small_1000_200")

# PERSIST_DIR=r"C:\Ritu Data\Upgrad\GenAI\RAG_Assignment\rag_legal\chroma_db_small_1000_200"
# os.makedirs(PERSIST_DIR, exist_ok=True)

# vector_store = Chroma(
#     collection_name="legal_docs",
#     embedding_function=embedding_function,
#     persist_directory=PERSIST_DIR,
# )

# print(f"Chunks to add: {len(chunks)}")
# # Stable, unique id per chunk (file path + character offset) so re-running does not create duplicates
# chunk_counter = Counter()
# chunk_ids = []
# for chunk in chunks:
#     rel_path = chunk.metadata['rel_path']
#     chunk.metadata['chunk_number'] = chunk_counter[rel_path]     # 0, 1, 2, ... within the file
#     chunk_counter[rel_path] += 1
#     chunk_ids.append(f"{rel_path}::{chunk.metadata['chunk_number']}::{chunk.metadata['start_index']}")

# # Set to a number (e.g. 2000) to test on a subset first; None embeds every chunk
# MAX_CHUNKS = None
# if MAX_CHUNKS:
#     chunks_to_add, ids_to_add = chunks[:MAX_CHUNKS], chunk_ids[:MAX_CHUNKS]
# else:
#     chunks_to_add, ids_to_add = chunks, chunk_ids

# print(f"Preparing to add {len(chunks_to_add)} chunks to the vector DB")
# print(f"{chunk_ids[:5]}")



In [ ]:
# SUPERSEDED by sections 2.3-2.5 (old Chroma DB count); kept for reference
# print("Chunks in vector DB:", vector_store._collection.count())

In [ ]:
# SUPERSEDED by sections 2.3-2.5 (old duplicate-id check); kept for reference
# # Skip chunks that are already stored (lets you resume after an interruption)
# try:
#     from chromadb.errors import DuplicateIDError
# except ImportError:   # older/newer chromadb versions expose a different exception
#     DuplicateIDError = ()

# def is_duplicate_id_error(error):
#     """True if the exception is Chroma's duplicate-id error."""
#     return isinstance(error, DuplicateIDError) or "duplicate" in f"{type(error).__name__} {error}".lower()

# existing_ids = set(vector_store.get(include=[])["ids"])

# # Drop chunks already in the DB, and repeated ids within this run (Chroma rejects a batch that contains the same id twice)
# pending_chunks, seen_ids, duplicates_skipped = [], set(existing_ids), 0
# duplicate_ids = []
# for chunk, chunk_id in zip(chunks_to_add, ids_to_add):
#     if chunk_id in seen_ids:
#         duplicates_skipped += 1
#         duplicate_ids.append(chunk_id)
#         continue
#     seen_ids.add(chunk_id)
#     pending_chunks.append((chunk, chunk_id))
# print(f"{len(existing_ids)} chunks already in DB, {duplicates_skipped} duplicate ids skipped, {len(pending_chunks)} to add")

# # Print the duplicate ids that were skipped
# if duplicates_skipped > 0:
#     print(f"Duplicate ids skipped: {duplicates_skipped}")
#     print("Duplicate ids:", duplicate_ids[:10], "..." if len(duplicate_ids) > 10 else "")
    


In [ ]:
# SUPERSEDED by sections 2.3-2.5 (old batched insert into the old Chroma DB); kept for reference
# BATCH_SIZE = 500
# failed_ids = []   # records skipped because of a duplicate id error from the vector store
# for batch_start in tqdm(range(0, len(pending_chunks), BATCH_SIZE), desc="Embedding batches"):
#     batch = pending_chunks[batch_start:batch_start + BATCH_SIZE]
#     try:
#         vector_store.add_documents(
#             documents=[chunk for chunk, chunk_id in batch],
#             ids=[chunk_id for chunk, chunk_id in batch],
#         )
#     except Exception as batch_error:
#         if not is_duplicate_id_error(batch_error):
#             raise
#         # The batch contains a duplicate id: retry record by record and skip only the offending ones
#         logger.warning("Duplicate id error in batch starting at %d; retrying records one by one", batch_start)
#         for chunk, chunk_id in batch:
#             try:
#                 vector_store.add_documents(documents=[chunk], ids=[chunk_id])
#             except Exception as record_error:
#                 if not is_duplicate_id_error(record_error):
#                     raise
#                 failed_ids.append(chunk_id)
#                 logger.warning("Skipping duplicate id: %s", chunk_id)

# print(f"Skipped {len(failed_ids)} records because of duplicate id errors")
# print("Chunks in vector DB:", vector_store._collection.count())

### **2.2 Create RAG Chain** <font color=red> [8 marks] </font><br>

#### **2.2.1** <font color=red> [5 marks] </font>
Form the complete RAG pipeline. 

You can either create a chain or directly the pipeline

In [ ]:
# Create a RAG chain
from urllib.parse import unquote
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

# Retriever: first find the contract named in the query, then search for the top-k similar chunks inside that file only.
# One short "header" per document (file name + opening text, where the title and parties usually appear) is TF-IDF indexed.
doc_paths = [document.metadata["rel_path"] for document in docs]
doc_headers = [re.sub(r"[-_.]", " ", unquote(document.metadata["file_name"][:-4])) + " " + document.page_content[:2000]
               for document in docs]
header_vectorizer = TfidfVectorizer(stop_words="english", sublinear_tf=True)
header_matrix = header_vectorizer.fit_transform(doc_headers)

## find_contract() returns the rel_path of the document that best matches the contract named in the query (the text before the first ';').
def find_contract(query):
    """rel_path of the document that best matches the contract named in the query (the text before the first ';').
    Returns None if the query does not name a contract (no ';') or nothing matches."""
    if ";" not in query:
        return None

    # Compute cosine similarity between the query and the document headers
    scores = linear_kernel(header_vectorizer.transform([query.split(";")[0]]), header_matrix)[0]
    return doc_paths[scores.argmax()] if scores.max() > 0 else None

# # OLD dense-only retriever (replaced by the hybrid BM25 + Cohere rerank retriever in section 2.2.1b)
# def retrieve(query, k=5):
#     """Top-k chunks, restricted to the matched contract when the query names one."""
#     rel_path = find_contract(query)
#     return vector_store.similarity_search(query, k=k, filter={"rel_path": rel_path} if rel_path else None)

# retriever = RunnableLambda(retrieve)

# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=OPENAI_API_KEY)

# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a legal assistant that answers questions about legal agreements "
     "(NDAs, commercial contracts, merger agreements and privacy policies). "
     "Answer ONLY from the context provided. If the context does not contain the answer, "
     "say that you could not find it in the provided documents. Be precise and concise."),
    ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"),
])

def format_docs(retrieved_docs):
    """Join retrieved chunks, labelling each with its source file."""
    return "\n\n".join(
        f"[Source: {retrieved_doc.metadata.get('file_name', 'unknown')}]\n{retrieved_doc.page_content}"
        for retrieved_doc in retrieved_docs
    )

answer_chain = prompt | llm | StrOutputParser()

# OLD chain assembly + quick test (moved to section 2.2.1b, after the new retriever is defined)
# The output is a dict with the question, the retrieved source documents and the answer.

# rag_chain = (
#     RunnableParallel(source_documents=retriever, question=RunnablePassthrough())
#     .assign(context=RunnableLambda(lambda chain_input: format_docs(chain_input["source_documents"])))
#     .assign(answer=answer_chain)
# )

# # Quick test (needs a valid API key and a populated vector DB)
# result = rag_chain.invoke("What is the term of the confidentiality obligations?")
# print(result["answer"])


#### **2.2.1b Improving retrieval: BM25 + Cohere reranking**

The dense-only retriever (commented out above) can miss clauses whose exact wording matters (names, defined terms, numbers). This section replaces it with a **hybrid retriever**:

1. Find the contract named in the query (same `find_contract` as before).
2. Get the top-`fetch_k` chunks from the **dense** vector search and from **BM25** keyword search, both restricted to that contract.
3. Merge the two ranked lists with **Reciprocal Rank Fusion (RRF)**.
4. **Rerank** the merged candidates with a Cohere cross-encoder and keep the top `k`.

If the query does not name a contract, the same steps run over the whole corpus.

In [ ]:
!pip install -q rank_bm25 langchain-cohere

In [ ]:
# Cohere API key (set COHERE_API_KEY in .env or the environment; you are asked for it here if missing)
import os
from getpass import getpass

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Enter Cohere API key: ")
COHERE_API_KEY = os.environ["COHERE_API_KEY"]


In [ ]:
# SUPERSEDED by sections 2.3-2.5 (old BM25 index over the 2000/500 chunks); kept for reference
# # BM25 indexes over the chunks (built lazily per contract and cached)
# from collections import defaultdict
# from rank_bm25 import BM25Okapi

# TOKEN_RE = re.compile(r"[a-z0-9]+")

# def tokenize(text):
#     return TOKEN_RE.findall(text.lower())

# def chunk_id_of(chunk):
#     return f"{chunk.metadata['rel_path']}::{chunk.metadata['chunk_number']}::{chunk.metadata['start_index']}"

# chunks_by_path = defaultdict(list)
# for chunk in chunks:
#     chunks_by_path[chunk.metadata["rel_path"]].append(chunk)

# _bm25_cache = {}

# def get_bm25(rel_path=None):
#     """(BM25 index, chunk list) for one contract, or for the whole corpus when rel_path is None."""
#     if rel_path not in _bm25_cache:
#         pool = chunks_by_path[rel_path] if rel_path else chunks
#         _bm25_cache[rel_path] = (BM25Okapi([tokenize(c.page_content) for c in pool]), pool)
#     return _bm25_cache[rel_path]

# def bm25_search(text, rel_path=None, k=20):
#     index, pool = get_bm25(rel_path)
#     scores = index.get_scores(tokenize(text))
#     top = scores.argsort()[::-1][:k]
#     return [pool[i] for i in top if scores[i] > 0]

# def rrf_fuse(ranked_lists, k=60):
#     """Reciprocal Rank Fusion: score = sum(1 / (k + rank)) over the lists; returns unique chunks, best first."""
#     scores, by_id = defaultdict(float), {}
#     for ranked in ranked_lists:
#         for rank, chunk in enumerate(ranked, start=1):
#             cid = chunk_id_of(chunk)
#             scores[cid] += 1.0 / (k + rank)
#             by_id.setdefault(cid, chunk)
#     return [by_id[cid] for cid in sorted(scores, key=scores.get, reverse=True)]


In [ ]:
# SUPERSEDED by sections 2.3-2.5 (old hybrid retriever and rag_chain over the 2000/500 chunks); kept for reference
# # Hybrid retriever: dense + BM25 -> RRF -> Cohere rerank
# from langchain_cohere import CohereRerank

# reranker = CohereRerank(model="rerank-english-v3.0", cohere_api_key=COHERE_API_KEY, top_n=5)

# def hybrid_retrieve(query, k=5, fetch_k=20):
#     """Top-k chunks using dense + BM25 search (inside the named contract when there is one), fused with RRF and reranked by Cohere."""
#     rel_path = find_contract(query)
#     # BM25 and the reranker work on the question only; the contract name before ';' is just noise for them
#     question_text = query.split(";", 1)[1].strip() if rel_path else query

#     dense_hits = vector_store.similarity_search(query, k=fetch_k, filter={"rel_path": rel_path} if rel_path else None)
#     bm25_hits = bm25_search(question_text, rel_path, fetch_k)
#     candidates = rrf_fuse([dense_hits, bm25_hits])[: 2 * fetch_k]

#     try:
#         reranker.top_n = k
#         return list(reranker.compress_documents(candidates, question_text))
#     except Exception as error:   # e.g. rate limit: fall back to the fused order
#         logger.warning("Cohere rerank failed (%s); using RRF order", error)
#         return candidates[:k]

# # C5 neighbour expansion: tried, recall did not improve much, reverted (see DECISION.md)
# # def expand_neighbours(hits, window=1):
# #     """Hits first, then the chunks next to them (same file, chunk_number +/- window), without duplicates."""
# #     out, seen = [], set()
# #     for hit in hits:                                    # the hits keep priority
# #         seen.add((hit.metadata["rel_path"], hit.metadata["chunk_number"])); out.append(hit)
# #     for hit in hits:
# #         pool = chunks_by_path[hit.metadata["rel_path"]]   # chunk_number == position in this list
# #         n = hit.metadata["chunk_number"]
# #         for j in range(max(0, n - window), min(len(pool), n + window + 1)):
# #             key = (hit.metadata["rel_path"], j)
# #             if key not in seen:
# #                 seen.add(key); out.append(pool[j])
# #     return out

# # def hybrid_retrieve_expanded(query, k=5):
# #     return expand_neighbours(hybrid_retrieve(query, k=k))

# # retriever = RunnableLambda(hybrid_retrieve_expanded)
# # rebuild rag_chain after this, as in the hybrid retriever cell

# # Use the hybrid retriever from here on (run_experiment and answer_question look up `retriever` / `rag_chain` at call time)
# retriever = RunnableLambda(hybrid_retrieve)

# # Chain: retrieve -> format context -> prompt -> LLM -> text (same structure as before, new retriever)
# rag_chain = (
#     RunnableParallel(source_documents=retriever, question=RunnablePassthrough())
#     .assign(context=RunnableLambda(lambda chain_input: format_docs(chain_input["source_documents"])))
#     .assign(answer=answer_chain)
# )

# # Quick test (needs valid OpenAI and Cohere keys and a populated vector DB)
# result = rag_chain.invoke("What is the term of the confidentiality obligations?")
# print(result["answer"])


### **2.3 Contextual headers and clause-aware chunking**

**Changes made in this section**
- The old fixed-size splitter (`RecursiveCharacterTextSplitter` 2000/500) and the old vector DB / BM25 / hybrid retriever cells are commented out (they are kept for reference).
- The text of each contract is now split on **clause boundaries** ("Article 5", "Section 2.1", "12.3 ..."), so one chunk holds one complete clause. Clauses shorter than 300 characters are merged into the next one; clauses longer than 3000 characters are split further without overlap. These clause-sized chunks are the **parents** (used in section 2.4).
- Every contract gets a short **contextual header** (`Contract: <file name> | Type: <folder> | Title: <first line>`). It is added to every searchable chunk in section 2.4, so a chunk from the middle of a contract still says which contract it belongs to.
- Offsets (`start_index`, `end_index`) are exact character positions in the original text, not guessed with `str.find` on the whole document, and every chunk id is unique by construction (`<file>::p<parent number>::c<child number>`).

In [ ]:
# 2.3 Contextual headers + clause-aware parent chunks
import re
from urllib.parse import unquote
from collections import Counter
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

PARENT_MIN_CHARS = 300     # clauses shorter than this are merged into the next clause
PARENT_MAX_CHARS = 3000    # clauses longer than this are split further (no overlap)

# A new clause starts at a line beginning with "Article 5", "Section 2.1", "12.", "12.3", "12.3.1" followed by text
HEADING_RE = re.compile(
    r"(?m)^[ \t]*(?:(?:ARTICLE|Article|SECTION|Section)[ \t]+[0-9IVXLivxl]+(?:\.[0-9]+)*[.:)]?"
    r"|[0-9]{1,2}(?:\.[0-9]{1,2}){0,3}[.)]?(?=[ \t]+[A-Z\"(]))"
)
oversize_splitter = RecursiveCharacterTextSplitter(
    chunk_size=PARENT_MAX_CHARS, chunk_overlap=0, separators=["\n\n", "\n", ". ", " ", ""])


def contract_header(doc):
    """One-line description of the contract, prepended to every searchable chunk of it."""
    name = re.sub(r"[-_.]+", " ", unquote(doc.metadata["file_name"].rsplit(".", 1)[0])).strip()
    title = next((line.strip() for line in doc.page_content.splitlines() if len(line.strip()) >= 8), "")[:100]
    return f"Contract: {name} | Type: {doc.metadata['folder']} | Title: {title}"


def split_into_parents(text):
    """(start, end) character ranges of clause-sized parents; text[start:end] is the parent."""
    bounds = sorted({0, len(text), *(match.start() for match in HEADING_RE.finditer(text))})
    ranges = []
    for start, end in zip(bounds[:-1], bounds[1:]):
        if ranges and ranges[-1][1] - ranges[-1][0] < PARENT_MIN_CHARS:
            ranges[-1] = (ranges[-1][0], end)            # previous clause is too small: extend it
        else:
            ranges.append((start, end))
    if len(ranges) > 1 and ranges[-1][1] - ranges[-1][0] < PARENT_MIN_CHARS:
        ranges[-2] = (ranges[-2][0], ranges[-1][1])      # last clause too small: merge into the previous one
        ranges.pop()

    sized = []
    for start, end in ranges:
        if end - start <= PARENT_MAX_CHARS:
            sized.append((start, end))
            continue
        cursor = start                                   # long clause: split it, walking forward through the text
        for piece in oversize_splitter.split_text(text[start:end]):
            index = text.find(piece, cursor)
            if index == -1 or index >= end:
                index = cursor
            sized.append((index, min(index + len(piece), end)))
            cursor = index + len(piece)

    trimmed = []                                         # drop leading/trailing whitespace, keep offsets exact
    for start, end in sized:
        raw = text[start:end]
        if raw.strip():
            start += len(raw) - len(raw.lstrip())
            trimmed.append((start, start + len(raw.strip())))
    return trimmed


headers = {}     # rel_path -> contextual header
parents = {}     # parent_id -> Document holding the full clause text (this is what the LLM receives)
for doc in docs:
    rel_path = doc.metadata["rel_path"]
    headers[rel_path] = contract_header(doc)
    for number, (start, end) in enumerate(split_into_parents(doc.page_content)):
        parent_id = f"{rel_path}::p{number}"
        text = doc.page_content[start:end]
        parents[parent_id] = Document(page_content=text, metadata={
            "rel_path": rel_path, "file_name": doc.metadata["file_name"], "folder": doc.metadata["folder"],
            "parent_id": parent_id, "start_index": start, "end_index": end,
            "section": text.split("\n", 1)[0][:80],
        })

# Sanity checks
parent_lengths = [len(parent.page_content) for parent in parents.values()]
print(f"{len(docs)} documents -> {len(parents)} clause-sized parents")
print(f"Parent length (chars): avg {sum(parent_lengths) / len(parent_lengths):.0f}, max {max(parent_lengths)}, min {min(parent_lengths)}")
print(Counter(parent.metadata["folder"] for parent in parents.values()))
first_parent = next(iter(parents.values()))
print("Header :", headers[first_parent.metadata["rel_path"]])
print("Section:", first_parent.metadata["section"])


### **2.4 Parent-child retrieval: small chunks for search, big chunks for answers**

**Changes made in this section**
- Each clause-sized parent from section 2.3 is split into small **child** chunks (500 characters, 100 overlap). The children are what is embedded and searched, so a short clause is not diluted by surrounding text.
- Every child's searchable text is `contextual header + section heading + child text`, used for both the embeddings and BM25.
- The children go into a **new Chroma folder** (`<PERSIST_DIR>_pc`, collection `legal_docs_pc`). The old DB is untouched. Re-embedding costs OpenAI credits; the insert can be resumed if interrupted.
- Search = dense + BM25 inside the contract found by `find_contract` (unchanged) -> Reciprocal Rank Fusion -> Cohere rerank of the **children**.
- The reranked children are then mapped back to their **parent** clauses (section 2.5), and the parents are what the LLM and Ragas see.

In [ ]:
# 2.4 Child chunks (small, searchable) built from the parents
CHILD_SIZE, CHILD_OVERLAP = 500, 100
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHILD_SIZE, chunk_overlap=CHILD_OVERLAP, separators=["\n\n", "\n", ". ", " ", ""])

child_chunks = []
for parent_id, parent in parents.items():
    cursor = 0
    for number, piece in enumerate(child_splitter.split_text(parent.page_content)):
        position = parent.page_content.find(piece, cursor)      # search only forward inside the parent
        if position == -1:
            position = cursor
        cursor = position + 1
        start = parent.metadata["start_index"] + position
        child_chunks.append(Document(
            # contextual header + section heading + text: this is what gets embedded and indexed by BM25
            page_content=f"{headers[parent.metadata['rel_path']]}\nSection: {parent.metadata['section']}\n{piece}",
            metadata={
                "rel_path": parent.metadata["rel_path"], "file_name": parent.metadata["file_name"],
                "folder": parent.metadata["folder"], "parent_id": parent_id, "section": parent.metadata["section"],
                "child_id": f"{parent_id}::c{number}", "start_index": start, "end_index": start + len(piece),
            }))
child_ids = [child.metadata["child_id"] for child in child_chunks]

child_lengths = [len(child.page_content) for child in child_chunks]
print(f"{len(parents)} parents -> {len(child_chunks)} children ({len(set(child_ids))} unique ids)")
print(f"Child length incl. header (chars): avg {sum(child_lengths) / len(child_lengths):.0f}, max {max(child_lengths)}, min {min(child_lengths)}")
print(child_chunks[0].page_content[:400])


In [ ]:
print(PERSIST_DIR)

In [ ]:
# 2.4 New vector DB for the child chunks (separate folder and collection; the old DB is untouched)
from langchain_chroma import Chroma
from tqdm.auto import tqdm

# PERSIST_DIR_PC = os.environ.get("PERSIST_DIR_PC") or (os.environ.get("PERSIST_DIR", "chroma_db_small").rstrip("\\/") + "_pc")
# PERSIST_DIR_PC="C:\\Ritu Data\\Upgrad\\GenAI\\RAG_Assignment\\rag_legal\\chroma_db_small_pc"
os.makedirs(PERSIST_DIR, exist_ok=True)
print("Child DB folder:", PERSIST_DIR)

vector_store_pc = Chroma(
    collection_name="legal_docs_pc",
    embedding_function=embedding_function,
    persist_directory=PERSIST_DIR,
)

MAX_CHILDREN = None      # set to a number (e.g. 2000) to test on a subset first; None embeds every child
existing_ids = set(vector_store_pc.get(include=[])["ids"])
candidates = list(zip(child_chunks, child_ids))[:MAX_CHILDREN] if MAX_CHILDREN else list(zip(child_chunks, child_ids))
pending = [(child, child_id) for child, child_id in candidates if child_id not in existing_ids]
print(f"{len(existing_ids)} children already in DB, {len(pending)} to add")

BATCH_SIZE = 500
for batch_start in tqdm(range(0, len(pending), BATCH_SIZE), desc="Embedding batches"):
    batch = pending[batch_start:batch_start + BATCH_SIZE]
    vector_store_pc.add_documents(documents=[child for child, _ in batch], ids=[child_id for _, child_id in batch])
print("Children in vector DB:", vector_store_pc._collection.count())


In [ ]:
# 2.4 Hybrid search over the children: dense + BM25 -> RRF -> Cohere rerank
from collections import defaultdict
from rank_bm25 import BM25Okapi
from langchain_cohere import CohereRerank

TOKEN_RE = re.compile(r"[a-z0-9]+")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

children_by_path = defaultdict(list)
for child in child_chunks:
    children_by_path[child.metadata["rel_path"]].append(child)

_bm25_pc_cache = {}

def get_bm25_pc(rel_path=None):
    """(BM25 index, child list) for one contract, or for the whole corpus when rel_path is None. Built lazily."""
    if rel_path not in _bm25_pc_cache:
        pool = children_by_path[rel_path] if rel_path else child_chunks
        _bm25_pc_cache[rel_path] = (BM25Okapi([tokenize(child.page_content) for child in pool]), pool)
    return _bm25_pc_cache[rel_path]

def bm25_search_pc(text, rel_path=None, k=30):
    index, pool = get_bm25_pc(rel_path)
    scores = index.get_scores(tokenize(text))
    return [pool[i] for i in scores.argsort()[::-1][:k] if scores[i] > 0]

def rrf_fuse_pc(ranked_lists, k=60):
    """Reciprocal Rank Fusion over lists of children; returns unique children, best first."""
    scores, by_id = defaultdict(float), {}
    for ranked in ranked_lists:
        for rank, child in enumerate(ranked, start=1):
            child_id = child.metadata["child_id"]
            scores[child_id] += 1.0 / (k + rank)
            by_id.setdefault(child_id, child)
    return [by_id[child_id] for child_id in sorted(scores, key=scores.get, reverse=True)]

reranker_pc = CohereRerank(model="rerank-english-v3.0", cohere_api_key=COHERE_API_KEY, top_n=10)

def search_children(query, child_k=10, fetch_k=30):
    """Top child_k children for the query (inside the contract named in it, when there is one)."""
    rel_path = find_contract(query)
    question_text = query.split(";", 1)[1].strip() if rel_path else query     # the contract name is noise for BM25/rerank
    dense_hits = vector_store_pc.similarity_search(query, k=fetch_k, filter={"rel_path": rel_path} if rel_path else None)
    bm25_hits = bm25_search_pc(question_text, rel_path, fetch_k)
    fused = rrf_fuse_pc([dense_hits, bm25_hits])[: 2 * fetch_k]
    try:
        reranker_pc.top_n = child_k
        return list(reranker_pc.compress_documents(fused, question_text))
    except Exception as error:      # e.g. rate limit: fall back to the fused order
        logger.warning("Cohere rerank failed (%s); using RRF order", error)
        return fused[:child_k]


### **2.5 Removing overlap duplicates and the final retriever**

**Changes made in this section**
- The reranked children are mapped back to their **parent** clauses. Several children of one clause (they overlap by design) would otherwise return the same clause repeatedly.
- `remove_overlap_duplicates` keeps the best-ranked passage and drops any later passage of the same file whose character span overlaps an already kept one by 50% or more of the shorter span.
- `retriever` and `rag_chain` are rebuilt on top of this. The LLM and the Ragas evaluation now receive up to `k` distinct clause-sized contexts.
- Use a new experiment name in the section 3 evaluation (for example `parent_child_b{n}`), so these results do not overwrite the earlier runs.

In [ ]:
# 2.5 Overlap de-duplication + final parent-child retriever
def remove_overlap_duplicates(docs, threshold=0.5):
    """Keep docs in rank order; drop a doc when >= threshold of the shorter span overlaps an already kept doc of the same file."""
    kept = []
    for doc in docs:
        start, end = doc.metadata["start_index"], doc.metadata["end_index"]
        is_duplicate = False
        for other in kept:
            if other.metadata["rel_path"] != doc.metadata["rel_path"]:
                continue
            overlap = min(end, other.metadata["end_index"]) - max(start, other.metadata["start_index"])
            shorter = min(end - start, other.metadata["end_index"] - other.metadata["start_index"])
            if overlap > 0 and shorter > 0 and overlap / shorter >= threshold:
                is_duplicate = True
                break
        if not is_duplicate:
            kept.append(doc)
    return kept


def children_to_parents(children):
    """Replace each child by its full clause (parent), keeping the rerank order and score."""
    return [
        Document(page_content=parents[child.metadata["parent_id"]].page_content,
                 metadata={**parents[child.metadata["parent_id"]].metadata,
                           "relevance_score": child.metadata.get("relevance_score")})
        for child in children
    ]


def parent_child_retrieve(query, k=5, child_k=10, fetch_k=30):
    """Search small chunks, answer with big ones: up to k distinct parent clauses."""
    children = search_children(query, child_k=child_k, fetch_k=fetch_k)
    return remove_overlap_duplicates(children_to_parents(children))[:k]


retriever = RunnableLambda(parent_child_retrieve)

# Chain: retrieve -> format context -> prompt -> LLM -> text (same structure as before, new retriever)
rag_chain = (
    RunnableParallel(source_documents=retriever, question=RunnablePassthrough())
    .assign(context=RunnableLambda(lambda chain_input: format_docs(chain_input["source_documents"])))
    .assign(answer=answer_chain)
)

# Quick test (needs valid OpenAI and Cohere keys and a populated child DB)
result = rag_chain.invoke("What is the term of the confidentiality obligations?")
print(result["answer"])
print([doc.metadata["parent_id"] for doc in result["source_documents"]])


#### **Recall comparison: dense only vs hybrid (BM25 + Cohere rerank)**
Runs the same 10 sampled questions (same seed) through the new retriever and compares context recall with the earlier dense-only run (`ragas_eval/experiments/retrieval_top5.csv`). Run this after the evaluation cells in section 3 (`run_experiment`, `dataset`).

In [ ]:
# Re-run the retrieval experiment with the hybrid retriever and compare with the dense-only run
# import run_experiment from ragas
# from ragas import run_experiment
# hybrid_results = (await run_experiment.arun(dataset, name="retrieval_hybrid_rerank")).to_pandas()
# dense_results = pd.read_csv("ragas_eval/experiments/retrieval_top5.csv")

# comparison = dense_results[["query", "context_recall"]].rename(columns={"context_recall": "dense_recall"}).merge(
#     hybrid_results[["query", "context_recall"]].rename(columns={"context_recall": "hybrid_recall"}), on="query", how="outer")
# comparison["query"] = comparison["query"].str[:80]
# print(comparison)
# print(f"\nMean context recall  dense only: {comparison['dense_recall'].mean():.3f}  |  hybrid + rerank: {comparison['hybrid_recall'].mean():.3f}")


#### **2.2.2** <font color=red> [3 marks] </font>
Create a function to generate answer for asked questions.

Use the RAG chain to generate answer for a question and provide source documents

In [ ]:
# Create a function for question answering
def answer_question(question, chain=rag_chain, show_output=True):
    """Answer a question with the RAG chain and return the answer, the retrieved chunks and their sources.

    Returns a dict with:
      question          - the question asked
      answer            - the generated answer
      source_documents  - retrieved chunks as LangChain Documents (most similar first)
      contexts          - the chunk texts only (the format Ragas expects)
      sources           - one entry per chunk: file path (benchmark-style, forward slashes) and character span
    """
    result = chain.invoke(question)

    sources = []
    for source_doc in result["source_documents"]:
        start = source_doc.metadata.get("start_index", 0)
        sources.append({
            "file_path": source_doc.metadata.get("rel_path", "unknown").replace("\\", "/"),
            "span": [start, start + len(source_doc.page_content)],
        })

    response = {
        "question": question,
        "answer": result["answer"],
        "source_documents": result["source_documents"],
        "contexts": [source_doc.page_content for source_doc in result["source_documents"]],
        "sources": sources,
    }

    if show_output:
        print(f"Question: {question}\n")
        print(f"Answer: {response['answer']}\n")
        print("Source documents:")
        for rank, (source, context) in enumerate(zip(sources, response["contexts"]), start=1):
            print(f"  {rank}. {source['file_path']}  chars {source['span'][0]}-{source['span'][1]}")
            print(f"     {context[:150].strip()!r}...")
    return response

In [ ]:
# Example question
question = "Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?"
response = answer_question(question)

In [ ]:
# Example question
question = "Consider DBT's Mutual Non-Disclosure Agreement; Does the document allow the Receiving Party to independently develop information that is similar to the Confidential Information?"
response = answer_question(question)

In [ ]:

# Example question
question = "Consider the Sponsored Research and License Agreement between ArTara, Inc. and The University of Iowa for TARA-002 Development; Is there a cap on liability under this contract?"
response = answer_question(question)

## **3. RAG Evaluation** <font color=red> [10 marks] </font><br>

### **3.1 Evaluation and Inference** <font color=red> [10 marks] </font><br>

#### **3.1.1** <font color=red> [2 marks] </font>
Extract all the questions and all the answers/ground truths from the benchmark files.

Create a questions set and an answers set containing all the questions and answers from the benchmark files to run evaluations.

In [ ]:
BENCHMARK_DIR="C:\\Ritu Data\\Upgrad\\GenAI\\RAG_Assignment\\rag_legal\\benchmarks_full"
print(f"Benchmark directory: {BENCHMARK_DIR}")

In [ ]:
# Create a question set by taking all the questions from the benchmark data
# Also create a ground truth/answer set
import json

# BENCHMARK_DIR = r"C:\Ritu Data\Upgrad\GenAI\RAG_Assignment\rag_legal\benchmarks"

# BENCHMARK_DIR = os.environ.get("BENCHMARK_DIR")
print(f"BENCHMARK_DIR: {BENCHMARK_DIR}")

BENCHMARK_NAMES = ["contractnli", "cuad", "maud"]

benchmark_data = []
for benchmark_name in BENCHMARK_NAMES:
    with open(os.path.join(BENCHMARK_DIR, f"{benchmark_name}.json"), encoding="utf-8") as benchmark_file:
        tests = json.load(benchmark_file)["tests"]
    for test in tests:
        snippets = test["snippets"]
        benchmark_data.append({
            "benchmark": benchmark_name,
            "question": test["query"],
            # Ground-truth snippets (file, span, answer): used for the retrieval metrics
            "ground_truth_snippets": [
                {"file_path": snippet["file_path"], "span": snippet["span"], "answer": snippet["answer"]}
                for snippet in snippets
            ],
            # One reference answer per question (all snippet answers joined): used by Ragas
            "ground_truth": "\n\n".join(snippet["answer"] for snippet in snippets),
        })

# Question set and answer (ground truth) set, in the same order
questions = [item["question"] for item in benchmark_data]
ground_truths = [item["ground_truth"] for item in benchmark_data]

# Sanity checks
print(f"Total questions: {len(questions)}  |  ground truths: {len(ground_truths)}")
print(Counter(item["benchmark"] for item in benchmark_data))
print(f"Questions with several ground-truth snippets: {sum(len(item['ground_truth_snippets']) > 1 for item in benchmark_data)}")
print(f"\nExample question: {questions[0]}")
print(f"Example ground truth: {ground_truths[0][:300]}")

In [ ]:
# Some benchmark snippets point to source files that are not in the corpus. Questions that depend only on
# such files can never be answered or "hit", so they are flagged and left out of the evaluation sample.
doc_rel_paths = {document.metadata["rel_path"].replace("\\", "/") for document in cleaned_docs}
corpus_paths = {path.replace("\\", "/") for path in doc_rel_paths}
for item in benchmark_data:
    item["answerable"] = any(snippet["file_path"] in corpus_paths for snippet in item["ground_truth_snippets"])

answerable_count = sum(item["answerable"] for item in benchmark_data)
print(f"Answerable questions: {answerable_count} of {len(benchmark_data)}  (not answerable: {len(benchmark_data) - answerable_count})")
print(Counter(item["benchmark"] for item in benchmark_data if not item["answerable"]))

#### **3.1.2** <font color=red> [5 marks] </font>
Create a function to evaluate the generated answers and retrieved contexts.

Evaluate the responses with *Ragas*. Additionally check the retrieval quality using 2 retrieval-driven metrics.

In [ ]:
!pip install -q ragas

In [ ]:
# ragas 0.4.x imports a Vertex AI class that langchain-community 0.4+ no longer ships.
# We don't use Vertex AI, so register an empty stand-in instead of downgrading langchain-community
# (a downgrade drags langchain-core/opentelemetry versions around and breaks the running kernel).
import sys
import types
if "langchain_community.chat_models.vertexai" not in sys.modules:
    vertexai_stub = types.ModuleType("langchain_community.chat_models.vertexai")
    vertexai_stub.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules["langchain_community.chat_models.vertexai"] = vertexai_stub


In [ ]:
# Function to evaluate the RAG pipeline (retrieval only: Ragas context precision and context recall)
import asyncio
from openai import AsyncOpenAI
from ragas import experiment, Dataset
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision, ContextRecall

# max_tokens is raised because long references (several snippets) overflow the default output limit
ragas_llm = llm_factory("gpt-4o-mini", client=AsyncOpenAI(api_key=OPENAI_API_KEY), max_tokens=8192)
precision_metric = ContextPrecision(llm=ragas_llm)
recall_metric = ContextRecall(llm=ragas_llm)

async def score_with_retry(metric, retries=3, **kwargs):
    """Return the metric score (float), retrying with a short backoff if the LLM call fails (e.g. truncated output).
    Returns None if all attempts fail."""
    for attempt in range(1, retries + 1):
        try:
            result = await metric.ascore(**kwargs)
            return result.value   # plain float; a MetricResult must not be used in a boolean test (it has no len())
        except Exception as error:
            if attempt == retries:
                print(f"Giving up after {retries} attempts: {error}")
                return None
            await asyncio.sleep(2 ** attempt)

@experiment()
async def run_experiment(row):
    # Retrieval only: no answer generation
    retrieved = retriever.invoke(row["query"])
    contexts = [doc.page_content for doc in retrieved]

    # Print the retrieved sources for debugging
    # print(f"Question: {row['query']}")
    # print("Retrieved sources:")
    # for rank, doc in enumerate(retrieved, start=1):
    #     print(f"  {rank}. {doc.metadata.get('rel_path', 'unknown')}  chars {doc.metadata.get('start_index', 0)}-{doc.metadata.get('start_index', 0) + len(doc.page_content)}")
    #     print(f"     {doc.page_content[:150].strip()!r}...")

    precision = await score_with_retry(
        precision_metric, user_input=row["query"], retrieved_contexts=contexts, reference=row["ground_truth"])
    recall = await score_with_retry(
        recall_metric, user_input=row["query"], retrieved_contexts=contexts, reference=row["ground_truth"])

    experiment_view = {
        **row,
        "retrieved_sources": [doc.metadata["rel_path"] for doc in retrieved],
        "context_precision": precision,
        "context_recall": recall,
    }
    return experiment_view


#### **3.1.3** <font color=red> [3 marks] </font>
Draw inferences by evaluating answers to questions.

To save time and computing power, you can just run the evaluation on 10 randomly sampled questions.

In [ ]:
import random
import pandas as pd

# Run the experiment on 10 randomly sampled benchmark queries
# random.seed(42)
# sample = random.sample(benchmark_data, 50)  # sample 50 questions for evaluation
# sample = benchmark_data[:1]  # for debugging: use the first question only
# Print the benchmark name, question and ground truth for each sampled query
# for item in sample:
#     print(f"Benchmark: {item['benchmark']}")
#     print(f"Question: {item['question']}")
#     print(f"Ground truth: {item['ground_truth'][:300]}...\n")
BATCH_SIZE = 5
# all answerable questions (or random.sample(...) for a larger subset)
all_rows = [{"query": i["question"], "ground_truth": i["ground_truth"], "benchmark": i["benchmark"]}
            for i in benchmark_data if i["answerable"]]

random.seed(42)
sample = all_rows if len(all_rows) <= 50 else all_rows[:50]  # for debugging: use all answerable questions
# sample = random.sample(all_rows, 50)

batch_results = []
for batch_no, start in enumerate(range(0, len(sample), BATCH_SIZE), start=1):
    batch_rows = sample[start:start + BATCH_SIZE]
    batch_ds = Dataset(name=f"retrieval_eval_b{batch_no}", backend="local/csv",
                       root_dir="ragas_eval", data=batch_rows)
    exp = await run_experiment.arun(batch_ds, name=f"hybrid_rerank_parent_chunking_b{batch_no}")   # waits for all 10 rows
    # exp = await run_experiment.arun(batch_ds, name=f"dense_only_with_contract{batch_no}")
    batch_results.append(exp.to_pandas())
    print(f"Batch {batch_no} done: ")
    print(batch_results[-1][["query", "context_recall", "context_precision"]])
    print(f"Batch {batch_no} done: mean recall {batch_results[-1]['context_recall'].mean():.3f}")
    print(f"Batch {batch_no} done: mean precision {batch_results[-1]['context_precision'].mean():.3f}")

results_all = pd.concat(batch_results, ignore_index=True)
print("Overall mean context recall:", results_all["context_recall"].mean())
print("Overall mean context precision:", results_all["context_precision"].mean())

In [ ]:
import pandas as pd

results_all = pd.concat(batch_results, ignore_index=True)
print("Overall mean context recall:", results_all["context_recall"].mean())
print("Overall mean context precision:", results_all["context_precision"].mean())

In [ ]:
results = results_all
print(f"Experiment results: {len(results_all)} rows")
print(results[["query", "context_precision", "context_recall"]])
# print(results[["query", "context_recall"]])
print(results[["context_precision", "context_recall"]].mean())

In [ ]:
# Example question
question = "Consider the Service Agreement between BicycleTX Ltd and Nigel Crockett for Chief Business Officer Position; Does this contract include any right of first refusal, right of first offer, or right of first negotiation?"
response = answer_question(question)

In [ ]:
# Full (untruncated) query text with its context recall
for position, row in results.iterrows():
    print(f"[{position}] context_recall = {row['context_recall']}")
    print(f"query: {row['query']}\n")


In [ ]:
print(results[["query"][0]][1])

In [ ]:
print(results[["context_recall"]].mean())

In [ ]:
## Steps for improving Recall
# Implemented in section 2.2.1b: hybrid search (dense + BM25, fused with RRF) followed by Cohere reranking.
# Original idea (kept for reference): Hybrid Search (Dense Vectors + BM25) -> top 50 -> cross-encoder reranker (Cohere Rerank or BGE-Reranker).

### **3.2 Evaluation without a hard contract filter**

**Purpose.** The benchmark questions have the form `<contract description>; <question>`, and `find_contract` uses the part before `;` to restrict the search to one file. A real user often does not name the contract, or names it loosely. This section measures the retriever without that file filter.

**Changes made in this section**
- Two question sets are built from the same queries as the main evaluation (`sample`), so results can be compared case by case:
  - **Soft name (no filter):** the text is kept but the `;` is removed, so `find_contract` returns `None` and the search runs over the whole corpus. The contract description is still in the query and can help dense/BM25 matching. This is the realistic "user mentions the contract loosely" case.
  - **Stripped (no name at all):** only the text after `;` is kept. The benchmark uses very few distinct question texts (templated questions), so most stripped questions are **identical for many contracts**. They are counted as *ambiguous*: no retriever can know which contract was meant, so this set only shows how much the contract name carries.
- Besides Ragas context recall / precision (soft-name set), a **file-level hit rate** is computed for both sets: whether the retrieved clauses come from a ground-truth file. It needs no LLM judge, and is not affected by Ragas crediting similar text from another contract.

In [ ]:
# 3.2 Build the question sets without a hard contract filter, from the same sample
question_to_item = {item["question"]: item for item in benchmark_data}

def strip_contract_name(question):
    """Only the question after the first ';' (the part before it names the contract)."""
    return question.split(";", 1)[1].strip() if ";" in question else question

def soften_contract_name(question):
    """Keep the contract description but drop the ';' so find_contract() finds no contract and does not filter."""
    return question.replace(";", " -", 1)

soft_rows, stripped_rows, gt_files_by_case = [], [], {}
for case_id, row in enumerate(sample):
    item = question_to_item[row["query"]]
    gt_files_by_case[case_id] = {snippet["file_path"].replace("\\", "/") for snippet in item["ground_truth_snippets"]}
    base = {"case_id": case_id, "ground_truth": row["ground_truth"], "benchmark": row["benchmark"]}
    soft_rows.append({**base, "query": soften_contract_name(row["query"])})
    stripped_rows.append({**base, "query": strip_contract_name(row["query"])})

# Ambiguity: the same question text asked about different contracts
text_counts = Counter(row["query"] for row in stripped_rows)
for row in stripped_rows:
    row["ambiguous"] = text_counts[row["query"]] > 1
for row, stripped in zip(soft_rows, stripped_rows):
    row["ambiguous"] = stripped["ambiguous"]
print(f"{len(sample)} cases | benchmarks: {dict(Counter(r['benchmark'] for r in sample))}")
print(f"Stripped set: {len(text_counts)} distinct question texts, {sum(r['ambiguous'] for r in stripped_rows)} ambiguous cases")
print(f"';' left in soft queries: {sum(';' in r['query'] for r in soft_rows)} (should be 0)")
print("\nORIGINAL:", sample[0]["query"][:160])
print("SOFT    :", soft_rows[0]["query"][:160])
print("STRIPPED:", stripped_rows[0]["query"][:160])


In [ ]:
# 3.2 File-level hit rate without the contract name (no LLM call except the reranker inside the retriever)
def file_hit_report(rows, k=5):
    records = []
    for row in rows:
        retrieved = retriever.invoke(row["query"])[:k]
        files = [doc.metadata["rel_path"].replace("\\", "/") for doc in retrieved]
        truth = gt_files_by_case[row["case_id"]]
        records.append({
            "case_id": row["case_id"], "benchmark": row["benchmark"], "ambiguous": row["ambiguous"],
            "hit_top1": bool(files) and files[0] in truth,                  # best result comes from a ground-truth file
            "hit_any": any(f in truth for f in files),                      # at least one result does
            "share_from_truth": sum(f in truth for f in files) / max(len(files), 1),
        })
    return pd.DataFrame(records)

for label, rows_to_check in (("SOFT name, no filter", soft_rows), ("STRIPPED, no name", stripped_rows)):
    file_hits = file_hit_report(rows_to_check)
    print(f"=== {label} ===")
    print(file_hits[["hit_top1", "hit_any", "share_from_truth"]].mean().round(3).to_string())
    print(file_hits.groupby("benchmark")[["hit_top1", "hit_any", "share_from_truth"]].mean().round(3).to_string())
    print()


In [ ]:
# 3.2 Ragas context recall / precision with the soft name and no filter (batched, like the main evaluation)
NC_BATCH_SIZE = 5
nc_batches = []
for batch_no, start in enumerate(range(0, len(soft_rows), NC_BATCH_SIZE), start=1):
    batch_rows = [{key: row[key] for key in ("case_id", "query", "ground_truth", "benchmark")}
                  for row in soft_rows[start:start + NC_BATCH_SIZE]]
    batch_ds = Dataset(name=f"softname_eval_b{batch_no}", backend="local/csv", root_dir="ragas_eval", data=batch_rows)
    exp = await run_experiment.arun(batch_ds, name=f"softname_parent_child_b{batch_no}")   # waits for the whole batch
    nc_batches.append(exp.to_pandas())
    print(f"Batch {batch_no}: recall {nc_batches[-1]['context_recall'].mean():.3f} | precision {nc_batches[-1]['context_precision'].mean():.3f}")

nocontract_results = pd.concat(nc_batches, ignore_index=True)
print("\nSoft name, no filter - mean context recall:", round(nocontract_results["context_recall"].mean(), 4),
      "| mean context precision:", round(nocontract_results["context_precision"].mean(), 4))
print("With filter (main run) - mean context recall:", round(results_all["context_recall"].mean(), 4),
      "| mean context precision:", round(results_all["context_precision"].mean(), 4))


## **4. Conclusion** <font color=red> [5 marks] </font><br>

### **4.1 Conclusions and insights** <font color=red> [5 marks] </font><br>

#### **4.1.1** <font color=red> [5 marks] </font>
**Conclusions and insights**

Every change and its result is logged in `DECISION.md`. Retrieval was measured with Ragas `ContextRecall` and `ContextPrecision` (judge `gpt-4o-mini`) on 50 queries per run.

**1. Results**

| Stage | Retriever | Recall | Precision |
| --- | --- | --- | --- |
| Baseline, no contract filter | Dense, k=5 | 0.687 | 0.629 |
| Baseline, contract filter | Dense, filtered by the contract named in the query, k=5 | 0.923 | 0.676 |
| Hybrid | Dense + BM25 -> RRF -> Cohere rerank | 0.950 | 0.792 |
| Improved chunking- benchmark data of 50 queries-small corpus | Headers + clause-aware parents + parent-child + overlap de-dup | 0.945 | 0.834 |
| Full corpus, same benchmark data| Same | 0.938 | 0.855 |
| Full corpus, 50 random queries from all benchmarks | Same as the improved-chunking row (child chunks -> RRF -> Cohere -> parents -> de-dup, k=5) | 0.930 | 0.708 |
| Same, soft name, no filter | Whole-corpus search | 0.915 | 0.826 |


**2. Insights about the data**
- The documents are long contracts with a clear clause structure (Article / Section / numbered clauses). Fixed-size chunks cut across clauses; chunking on clause boundaries keeps a clause together.
- The benchmark questions are templated (`<contract description>; <question>`): only 17, 35 and 19 distinct question texts for contractnli, cuad and maud. The question alone does not identify the contract, so the contract name carries most of the signal.
- Many contracts are near-duplicates (shared boilerplate). This is why choosing the right file is hard, and why a bigger corpus makes it harder.
- The first 50 evaluation cases were all contractnli (short NDAs), the easiest benchmark. Scores on that sample overstate performance on CUAD and MAUD.

**3. Insights about the pipeline**
- **Contract-level filtering is the biggest single gain.** Recall went from 0.687 to 0.923. Once the right file is known, retrieval becomes a within-document problem.
- **Hybrid search + reranking** added about +0.03 recall and +0.12 precision. Legal text has exact terms (defined terms, section numbers) that BM25 matches and embeddings blur; RRF merges both lists and Cohere orders the final candidates.
- **Chunking changed precision more than recall.** Contextual headers, clause-sized parents and small child chunks for search raised precision from 0.792 to 0.834 while recall stayed about the same. These three changes were evaluated together, so the effect of each one is not separated.
- **Fixing chunk-id collisions matters for correctness.** Ids built from `str.find` repeated for repeated text, and the old insert silently skipped duplicates, so some chunks were never embedded. The new ids use exact offsets and are unique.
- **Things that did not help:** neighbour expansion (recall did not improve, reverted) and lowering the final k (both metrics fell).

**4. Insights about the RAG process and the results**
- **Retrieval failures are mostly routing and selection failures, not generation failures.** When the file was right but the clause was missing, the cause was chunking; when the file was wrong, nothing downstream could recover.
- **Without the contract name the pipeline is not reliable.** With the description kept but no filter, recall fell by 0.03 (0.945 -> 0.915). With no name at all, the file-level hit rate was at chance (`hit_top1` 0.020, `hit_any` 0.040). Because the questions are templated, this is a property of the benchmark and no retriever could do better.
- **More data in the vector database did not hurt retrieval.** With the same retriever, the full corpus gave 0.930 / 0.708 on 50 random queries, but 0.938 / 0.855 when the benchmark was run without random sampling. That is above the small-corpus result (0.945 / 0.834) on precision and within noise on recall. The low precision of the random run came from that particular sample of 50 queries, not from the larger corpus. This also shows how much a 50-query sample can swing precision (0.708 vs 0.855).
- **Ragas scores are noisy.** With 50 queries and an LLM judge, differences of about 0.02-0.05 should not be read as real.

**5. Limitations and next steps**
- Evaluation is retrieval only; faithfulness and answer correctness have not been measured.
- Add a per-benchmark breakdown and a file-routing check for the full-corpus run to see which benchmarks and contracts still score low.
- Evaluate contextual headers, parent-child and de-duplication one at a time.
- Improve contract routing (for example a reranked or hybrid file-level search instead of one TF-IDF match) so that a wrong file does not zero out a query.
- Use a fixed random seed and a stratified sample across benchmarks so that runs are comparable.
